In [1]:
from Montreal_UHI_toolbox import *
from sklearn.linear_model import LinearRegression

In [ ]:
#Load all summer data from observation and otherwise into stations dataset

summer = {}
summer_std = {}
path = '/runoff/gulley/St_Laurent/intermediates'
season = 'JJA'
for field in ['tasmin','tasmax','tasavg']:

    # Loading simulated summer averages 
    summer[f'{field}_C'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_avg_{field}_noTEB.zarr')[field].sel(season='JJA') - 273.15
    summer[f'{field}_T'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_avg_{field}_TEB.zarr')[field].sel(season='JJA') - 273.15

    # Loading simulated summer standard deviations
    summer_std[f'{field}_C'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_std_{field}_noTEB.zarr')[field].sel(season='JJA')
    summer_std[f'{field}_T'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_std_{field}_TEB.zarr')[field].sel(season='JJA')

    
    # Adding smoothed simulated summer averages to nearest station
    stations = add_blurred_field_to_stations(summer[f'{field}_C'],stations=stations,name=f'{field}_summer_avg_C')
    stations = add_blurred_field_to_stations(summer[f'{field}_T'],stations=stations,name=f'{field}_summer_avg_T')

    # Adding smoothed summer standard deviation to nearest station
    stations = add_blurred_field_to_stations(summer_std[f'{field}_C'],stations=stations,name=f'{field}_summer_std_C')
    stations = add_blurred_field_to_stations(summer_std[f'{field}_T'],stations=stations,name=f'{field}_summer_std_T')

# Loading station summer averages
tasmax_S = xr.open_zarr(f'{path}/station/seasons_avg_tasmax.zarr')['tasmax'].sel(season=season)
tasmin_S = xr.open_zarr(f'{path}/station/seasons_avg_tasmin.zarr')['tasmin'].sel(season=season)
tasavg_S = xr.open_zarr(f'{path}/station/seasons_avg_tas.zarr')['tas'].sel(season=season).rename('tasavg')
# Loading station summer standard deviations
tasmax_std_S = xr.open_zarr(f'{path}/station/seasons_std_tasmax.zarr')['tasmax'].sel(season=season)
tasmin_std_S = xr.open_zarr(f'{path}/station/seasons_std_tasmin.zarr')['tasmin'].sel(season=season)
tasavg_std_S = xr.open_zarr(f'{path}/station/seasons_std_tas.zarr')['tas'].sel(season=season).rename('tasavg')


for da in [tasmax_S,tasmin_S,tasavg_S]:
    stations[f'{da.name}_summer_avg_S'] = da

for da in [tasmax_std_S,tasmin_std_S,tasavg_std_S]:
    stations[f'{da.name}_summer_std_S'] = da

stations = stations.dropna(dim="station", subset=['tasmax_summer_avg_S','tasmin_summer_avg_S','tasavg_summer_avg_S'])

In [16]:
stations_to_exclude = ['POINTE AU CHENE', 'NAMINIGUE', 'HUBERDEAU','VALLEYFIELD','STE MADELEINE','SAINT-GERMAIN-DE-GRANTHAM','ST GUILLAUME','MACDONALD COLLEGE','DRUMMONDVILLE','BROME','ST TITE','ST COME','BERTHIERVILLE','MORRISBURG']
stations = stations.where(~stations.station_name.isin(stations_to_exclude), drop=True)

In [20]:
for model_suffix, model_title in zip(['S','C','T'],['Observed','CLASS','TEB+CLASS']):
    fig = go.Figure()
    names = stations.station_name.values
    x = stations.urban_fraction_blurred_std1p5.values
    lat = stations.lat.values

    for coord,colour,subscript,symbol,line_style in zip([f'tasmax_summer_avg_{model_suffix}',f'tasavg_summer_avg_{model_suffix}',f'tasmin_summer_avg_{model_suffix}'],
    ['gold','orange','brown'],
    ['max','avg','min'],
    ['diamond', 'circle', 'square'],
    ['dash', 'dashdot', 'dot']):

        y = stations[coord].values
        
        # Scatter stations
        fig.add_trace(go.Scatter(
            x=x,
            y=y,
            mode='markers',
            text=names,
            textposition='top center',
            marker=dict(
                size=8,
                symbol=symbol,
                color=lat,                 
                colorscale='Viridis',
            ),
            name = fr'$\overline{{T}}^{{\text{{JJA}}}}_{{\text{{{subscript}}}}}$',
            customdata=np.stack([lat], axis=-1),
            hovertemplate = (
                f'%{{text}}'
                f'<br>lat: %{{customdata[0]:.2f}}°'
                f'<br>urban_blurred_std1p5: %{{x:.2f}}'
                f'<br>{coord}: %{{y:.2f}}°C'
                f'<extra></extra>'
            )
        ))

        # Colourbar
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(
                colorscale='Viridis',
                cmin=min(lat),
                cmax=max(lat),
                colorbar=dict(
                    title='Latitude (°N)',
                    orientation='h',
                    xanchor='left',
                    yanchor='bottom',
                    y=0.05,
                    x = 0.6,
                    len = 0.3,
                    thickness=15
                ),
                showscale=True
            ),
            hoverinfo='none',
            
            showlegend=False
        ))

        # Linear Regression
        model = LinearRegression()
        X = x.reshape(-1, 1)
        model.fit(X,y)
        r_square = model.score(X,y)
        xline = np.linspace(0,1,2)
        yline = xline*model.coef_ + model.intercept_
        line_label = f'$y={round(model.coef_[0]*10)/10}x+{round(model.intercept_*10)/10}, R^2={round(r_square*100)/100}$'
        
        # Adding best-fit line
        # fig.add_trace(go.Scatter(
        #     x=xline,
        #     y=yline.flatten(),
        #     mode='lines',
        #     line=dict(color=colour,dash='dot'),
        #     name=line_label
        # ))
        
        fig.add_trace(go.Scatter(
            x=xline,
            y=yline.flatten(),
            mode='lines',
            line=dict(color='black', dash=line_style),
            name=line_label
        ))
        
        # Axes labels, title, limits
        fig.update_layout(
            title=f'{model_title} Average Summer Daily Temperatures',
            xaxis_title='Urban Fraction',
            yaxis_title='Temperature (°C)',
            xaxis=dict(range=[-0.04, 1.01]),
            yaxis=dict(range=[9,31])
        )

    # Urban thresholds
    urban_thresh = 0.5
    suburban_thresh = 0.2
    rural_thresh = 0.01

    # Marking thresholds
    fig.add_vline(x=rural_thresh, line=dict(color='grey', dash='dash'), annotation_text='Rural', annotation_position='top left',annotation_font=dict(color='grey'))
    fig.add_vline(x=urban_thresh, line=dict(color='grey', dash='dash'), annotation_text='Urban', annotation_position='top right',annotation_font=dict(color='grey'))
    suburban_x = (rural_thresh + urban_thresh) / 2
    fig.add_annotation(
        x=suburban_x,
        y=1.0,
        yref='paper',
        text='Suburban',
        showarrow=False,
        align='center',
        font=dict(color='grey')
    )


    # Display figure
    fig.update_xaxes(dtick=0.1)
    # fig.show()
    fig.write_html(f'info/plots/average_summer_daily_temp_vs_urban_fraction_{model_suffix}.html',include_mathjax='cdn')